# Transformer 架构
Transformer 是由 Google 在 2017 年论文 《Attention Is All You Need》 中提出的一种深度学习架构。它目前是所有主流大语言模型（如 ChatGPT、Claude）的核心技术。

## 1. 核心原理：注意力机制（Self-Attention Mechanism）
自注意力机制是 Transformer 的“灵魂”。它允许模型在处理序列中的某个元素时，能够同时观察并评估序列中其他所有元素的重要性。

* 功能：在处理一个词时，它能够同时观察句子中的所有词，并根据重要性分配不同的“权重”。

* 直观理解：例如句子 “动物没过马路，因为它太累了”。当模型处理“它”这个词时，自注意力机制会识别出“它”与“动物”的关系比与“马路”的关系更强，从而准确理解语义。

### 三个角色

| 角色 | 全称 | 含义 |
|------|------|------|
| Q | Query（查询） | 这个词在"问"什么——我需要什么信息？ |
| K | Key（键） | 每个词在"广播"什么——我能提供什么信息？ |
| V | Value（值） | 每个词实际携带的内容——我的具体信息是什么？ |

### 计算四步

```
Q × Kᵀ   →  相似度分数
  ÷ √d_k  →  缩放（防止数值太大导致梯度消失）
  softmax  →  转成概率（每行权重加和 = 1）
  × V      →  加权求和，得到输出
```
> **为什么除以 √d_k？**
> d_k 很大时点积结果会爆炸，softmax 饱和后梯度趋近于 0，模型无法学习。除以 √d_k 把数值压回合理范围。

### 多头注意力（Multi-Head Attention）

把 Q/K/V 分成 h 组（原论文 h=8），每组独立计算 Attention，最后拼接。
不同的"头"关注不同维度的语义——有的关注语法，有的关注语义，有的关注指代关系。


## 2. 架构组成
标准的 Transformer 由以下两个主要模块构成：

### A. 编码器 (Encoder)
* 作用：负责“读懂”输入信息。

* 过程：将输入的文本转换为包含丰富上下文信息的数学向量（特征表示）。

* 应用：典型模型如 BERT。

```
输入 Token → Embedding + 位置编码
    ↓
┌──────────────────────────────┐
│  Multi-Head Self-Attention   │  ← 双向，能看全句
│  Add & Norm（残差连接）       │
│  Feed-Forward Network        │
│  Add & Norm                  │
└──────────────────────────────┘
        × 6 层
    ↓
上下文向量（每个词都包含了全局信息）
```

### B. 解码器 (Decoder)
* 作用：负责“生成”输出结果。

* 过程：基于编码器提供的上下文以及已生成的上文，逐个预测并输出下一个词。

* 应用：典型模型如 GPT。

```
已生成的 Token → Embedding + 位置编码
    ↓
┌──────────────────────────────────┐
│  Masked Self-Attention           │  ← 单向，只能看左边（因果掩码）
│  Add & Norm                      │
│  Cross-Attention                 │  ← Q 来自 Decoder，K/V 来自 Encoder
│  Add & Norm                      │
│  Feed-Forward Network            │
│  Add & Norm                      │
└──────────────────────────────────┘
        × 6 层
    ↓
softmax → 下一个词的概率分布

### 关键差异对比

| | Encoder | Decoder |
|---|---------|---------|
| 注意力方向 | 双向（看全文） | 单向（只看前文） |
| 掩码机制 | 无 | 因果掩码（下三角矩阵） |
| 额外子层 | 无 | Cross-Attention（连接 Encoder） |
| 训练目标 | 理解语义 | 预测下一个词 |
| 擅长任务 | 分类、NER、语义相似度 | 对话、续写、代码生成 |
| 代表模型 | BERT、RoBERTa | GPT、Claude、LLaMA |

> **为什么现代大模型都选 Decoder-only？**
> 结构更简单，规模扩展更容易。生成任务可以覆盖几乎所有 NLP 场景——
> 理解、翻译、问答都可以转化为"根据输入生成输出"的形式。

## 3. 位置编码——解决"不知道词序"的问题

Self-Attention 本身**没有位置感知**，"猫追狗"和"狗追猫"对它来说完全一样。

**解决方案**：在 Embedding 上直接加一个位置向量。

```
最终输入 = 词向量 + 位置编码向量
```

原始论文用 sin/cos 函数，不同维度使用不同频率：

| 维度范围 | 频率 | 作用 |
|---------|------|------|
| 低维（0–63） | 高频，变化快 | 区分相邻位置（类比秒针） |
| 中维（256–319） | 中频 | 区分中等距离（类比分针） |
| 高维（448–511） | 低频，变化慢 | 区分远距离位置（类比时针） |

> **类比时钟**：秒针 + 分针 + 时针的组合唯一标识一天中的每个时刻，
> 位置编码用同样的思路唯一标识序列中的每个位置。

**现代改进 —— RoPE（旋转位置编码）**
LLaMA、Claude 等模型用 RoPE 替代原始 sin/cos 编码。
优势：可以**外推到训练时没见过的更长序列**，这是大模型支持 128k、200k 超长上下文的技术基础之一。

## 4. 两个让 Transformer 能堆很深的设计

### 残差连接（Residual Connection）

每个子层的公式：`output = LayerNorm(x + f(x))`，而不是 `output = f(x)`

**意义**：
- 梯度可以直接跳过子层回传 → 解决深层网络梯度消失
- 最坏情况 = 恒等映射（什么都不学）→ 不会变差
- 原论文堆 6 层，现代大模型堆到 96 层，靠的就是这个设计

### LayerNorm（层归一化）

在每个子层后把数值分布拉回均值 0、方差 1 附近。
**作用**：防止数值爆炸，让训练过程更稳定。

## 5. 模型对比：Transformer vs 传统模型

|特性 | 传统模型 (RNN/LSTM) | Transformer|
| :--- | :----: | ---: |
|处理方式 | 顺序处理（一个词接一个词）| 并行处理（全句同时输入）|
|计算效率 | 慢，难以利用 GPU 加速 | 极快，适合大规模训练|
|记忆长度 | 容易“遗忘”长距离信息 | 长程依赖处理能力强|
|位置感知 | 依靠输入顺序自然感知 | 依靠 位置编码 (Positional Encoding)|

> **RNN 的根本问题**：信息必须经过每一个时间步传递，句子越长丢失越严重。
> Transformer 让任意两个词**直接计算关联**，彻底绕开了这个瓶颈。

## 6. 原始论文关键超参数

| 参数 | 数值 | 含义 |
|------|------|------|
| N（层数） | 6 | Encoder 和 Decoder 各 6 层 |
| d_model | 512 | 向量维度 |
| h（注意力头数） | 8 | 多头注意力的头数 |
| d_k = d_v | 64 | 每个头的维度（512 ÷ 8） |
| d_ff | 2048 | FFN 隐层维度（d_model × 4） |
| 总参数量 | 65M | 对比 GPT-3 的 175B |

## 7. 行业影响

开启大模型时代的原因只有一个：**支持大规模并行训练**。
研究人员可以把整个互联网的文本塞进去，模型越大、数据越多、效果越好（Scaling Law）。

现在已远超 NLP 范围：

| 领域 | 应用 | 思路 |
|------|------|------|
| 计算机视觉 | ViT | 把图像切成 16×16 小块，当成"词"输入 |
| 语音识别 | Whisper | 把音频频谱当成序列处理 |
| 生物信息学 | AlphaFold2 | 把氨基酸序列当成"语言"来理解 |
| 多模态 | GPT-4o、Gemini | 文字、图片、音频统一用 Transformer 处理 |

> 如果把 AI 比作一座摩天大楼，Transformer 就是那套可以无限向上加层、
> 且越高越稳固的钢结构框架。


## 一、Transformer 整体架构
原始论文（2017）的关键数字：

* Encoder × 6 层，每层 2 个子层：Multi-Head Self-Attention + FFN
* Decoder × 6 层，每层 3 个子层：Masked Self-Attention + Cross-Attention + FFN
* d_model = 512，h = 8 个注意力头，d_k = 64，d_ff = 2048

## 二、Self-Attention 核心公式

In [5]:
import numpy as np

def scaled_dot_product_attention(Q, K, V):
    """
    Q: 查询矩阵 (seq_len, d_k) — "我在找什么"
    K: 键矩阵   (seq_len, d_k) — "我能提供什么"
    V: 值矩阵   (seq_len, d_k) — "我的实际内容"
    """
    d_k = Q.shape[-1]

    # 步骤1：Q 和 K 做点积，计算相似度分数
    scores = Q @ K.T                    # (seq_len, seq_len)

    # 步骤2：除以 sqrt(d_k) 防止梯度消失
    # d_k 较大时点积值很大，softmax 会饱和
    scores = scores / np.sqrt(d_k)

    # 步骤3：softmax 归一化成概率分布（每行加和=1）
    exp_scores = np.exp(scores - scores.max(axis=-1, keepdims=True))
    weights = exp_scores / exp_scores.sum(axis=-1, keepdims=True)

    # 步骤4：用权重加权求和 V，得到输出
    output = weights @ V                # (seq_len, d_k)

    return output, weights

# 用小例子感受一下
np.random.seed(42)
seq_len, d_k = 4, 8
Q = np.random.randn(seq_len, d_k)
K = np.random.randn(seq_len, d_k)
V = np.random.randn(seq_len, d_k)

output, weights = scaled_dot_product_attention(Q, K, V)
print("注意力权重矩阵（每行加和=1）：")
print(weights.round(3))
print("输出形状：", output.shape)       # (4, 8) 和输入一样

注意力权重矩阵（每行加和=1）：
[[0.084 0.255 0.515 0.145]
 [0.641 0.133 0.017 0.209]
 [0.47  0.088 0.111 0.331]
 [0.178 0.492 0.201 0.13 ]]
输出形状： (4, 8)


## 三、Encoder vs Decoder 核心差异

In [6]:
# 用代码直观理解最关键的区别：掩码

import numpy as np

def create_causal_mask(seq_len):
    """
    Decoder 用的因果掩码（下三角矩阵）
    生成第 t 个词时，只能看到位置 0 到 t-1 的词
    """
    # 下三角为 True（可见），上三角为 False（遮蔽）
    mask = np.tril(np.ones((seq_len, seq_len), dtype=bool))
    return mask

# Encoder：双向，所有位置互相可见
encoder_mask = np.ones((5, 5), dtype=bool)
print("Encoder 掩码（全可见）：")
print(encoder_mask.astype(int))

# Decoder：单向，只能看左边
decoder_mask = create_causal_mask(5)
print("\nDecoder 因果掩码（只看前文）：")
print(decoder_mask.astype(int))

# 输出：
# Encoder：全是 1
# Decoder：
# 1 0 0 0 0
# 1 1 0 0 0
# 1 1 1 0 0
# 1 1 1 1 0
# 1 1 1 1 1

Encoder 掩码（全可见）：
[[1 1 1 1 1]
 [1 1 1 1 1]
 [1 1 1 1 1]
 [1 1 1 1 1]
 [1 1 1 1 1]]

Decoder 因果掩码（只看前文）：
[[1 0 0 0 0]
 [1 1 0 0 0]
 [1 1 1 0 0]
 [1 1 1 1 0]
 [1 1 1 1 1]]


|Encoder | Decoder|
|------|------|
|注意力方向 | 双向（看全文）| 单向（只看前文）|
|掩码 | 无 | 因果掩码（下三角）|
|擅长任务 | 理解（分类、NER）| 生成（对话、续写）|
|代表模型 | BERT | GPT、Claude、LLaMA|

## 四、位置编码

In [7]:
import numpy as np
import matplotlib.pyplot as plt

def positional_encoding(seq_len, d_model):
    """
    原始论文的 sin/cos 位置编码
    不是可学习参数，是纯数学公式生成的固定值
    """
    # 初始化位置编码矩阵
    PE = np.zeros((seq_len, d_model))

    # pos：词在序列中的位置（0, 1, 2, ...）
    pos = np.arange(seq_len).reshape(-1, 1)       # (seq_len, 1)

    # i：向量的维度索引（0, 2, 4, ... 偶数维用 sin，奇数维用 cos）
    i = np.arange(0, d_model, 2)                  # (d_model/2,)

    # 分母：10000^(2i/d_model)，维度越高频率越低
    div = np.power(10000, i / d_model)

    PE[:, 0::2] = np.sin(pos / div)   # 偶数维用 sin
    PE[:, 1::2] = np.cos(pos / div)   # 奇数维用 cos

    return PE

PE = positional_encoding(seq_len=50, d_model=512)
print("位置编码矩阵形状：", PE.shape)   # (50, 512)
print("第 0 个位置，前 8 维：", PE[0, :8].round(3))
print("第 1 个位置，前 8 维：", PE[1, :8].round(3))

# 直觉：
# 低维（i=0）：sin 的周期很短，相邻位置值差异大 → 区分近距离位置
# 高维（i=255）：sin 的周期极长，整个序列内几乎不变 → 区分远距离位置
# 组合所有维度 → 唯一标识每个位置

位置编码矩阵形状： (50, 512)
第 0 个位置，前 8 维： [0. 1. 0. 1. 0. 1. 0. 1.]
第 1 个位置，前 8 维： [0.841 0.54  0.822 0.57  0.802 0.597 0.782 0.623]


为什么不用整数 0, 1, 2, 3？

* 整数会随序列变长而数值爆炸，难以归一化
* sin/cos 的值域固定在 [-1, 1]，稳定
* 最重要：PE(pos+k) 可以用 PE(pos) 的线性变换得到，让模型能学到相对位置关系

## 五、残差连接 + LayerNorm

In [8]:
import numpy as np

def layer_norm(x, eps=1e-6):
    """
    层归一化：对每个样本的特征维度做标准化
    稳定训练，加速收敛
    """
    mean = x.mean(axis=-1, keepdims=True)
    std = x.std(axis=-1, keepdims=True)
    return (x - mean) / (std + eps)

def transformer_sublayer(x, sublayer_fn):
    """
    残差连接 + LayerNorm
    公式：output = LayerNorm(x + sublayer(x))

    残差连接的意义：
    1. 梯度可以直接跳过子层回传 → 解决深层网络梯度消失
    2. 最坏情况 = 恒等映射（什么都不学）→ 不会变差
    3. 支持堆叠几十/百层
    """
    return layer_norm(x + sublayer_fn(x))   # x + f(x)，不是 f(x)

# 示意
x = np.random.randn(4, 512)                   # 4 个 token，每个 512 维
attention_output = lambda x: x * 0.1         # 模拟 attention 子层
result = transformer_sublayer(x, attention_output)
print("输出形状：", result.shape)              # (4, 512)，形状不变

输出形状： (4, 512)


## 六、现代 LLM 的架构演变

为什么现代大模型都选 Decoder-only？

* 规模扩展更容易，越大越强
* Few-shot 能力更强（不需要针对每个任务微调）
* 生成任务覆盖了几乎所有 NLP 场景（理解任务也可以转化为生成）